# Step-1: Preprocessing - Clean, Split(train, test)

In [39]:
import pandas as pd

# Load dataset and strip leading/trailing spaces from column names
df = pd.read_csv("./data.csv")
df.columns = df.columns.str.strip()

# Check shape (must be >= 500 rows and >= 12 features) and class balance
# Dataset has 6819 rows and 96 columns (95 features + 1 target 'Bankrupt?')
print("Dataset Shape:", df.shape)
print("Target Distribution:\n", df["Bankrupt?"].value_counts())

Dataset Shape: (6819, 96)
Target Distribution:
 Bankrupt?
0    6599
1     220
Name: count, dtype: int64


In [40]:
target_col = "Bankrupt?"

# Partition dataset by binary target: Bankrupt (1, 220 instances) and Non-Bankrupt (0, 6599 instances).
bankrupt_df = df[df[target_col] == 1]
non_bankrupt_df = df[df[target_col] == 0]

In [41]:
from sklearn.model_selection import train_test_split

# Split Bankrupt: 70% to train (154 instances), 30% to test (66 instances).
b_train, b_test = train_test_split(bankrupt_df, test_size=0.30, random_state=99)

# Split Non-Bankrupt: 30% to train (4619 instances), 70% to test (1980 instances).
nb_train, nb_test = train_test_split(non_bankrupt_df, test_size=0.70, random_state=99)

# Combine splits to produce train_data (2134 rows) and test_data (4685 rows).
train_df = pd.concat([b_train, nb_train]).sample(frac=1, random_state=99).reset_index(drop=True)
test_df = pd.concat([b_test, nb_test]).sample(frac=1, random_state=99).reset_index(drop=True)

In [42]:
# Export both datasets with features and target preserved for model training and Streamlit evaluation.
train_df.to_csv("train_data.csv", index=False)
test_df.to_csv("test_data.csv", index=False)

# Step-2: Model Training

In [43]:
import os

os.makedirs('./model', exist_ok=True)

In [44]:
import pandas as pd

# Read the saved train dataset directly
train_df = pd.read_csv("./train_data.csv")

target_col = "Bankrupt?"
X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

In [45]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

# Define the 5 mandatory classification models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=99),
    "Decision Tree": DecisionTreeClassifier(random_state=99),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": SVC(probability=True, random_state=99)
}

In [46]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib

# Fit and persist each pipeline
for name, clf in models.items():
    pipeline = Pipeline([
        ("var_thresh", VarianceThreshold(threshold=0.0)),
        ("scaler", StandardScaler()),
        ("classifier", clf)
    ])
    
    pipeline.fit(X_train, y_train)
    
    filename = name.lower().replace(" ", "_") + ".joblib"
    joblib.dump(pipeline, os.path.join("model", filename))

/Users/thedeveloperswork/Workspace/GitHub/2026AJ05314/ml_assignment_2/.venv/lib/python3.15/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


# Step-3: Model Evaluation and Performance Metrics

In [47]:
import pandas as pd

# Load test_data.csv and isolate features from ground-truth target
test_df = pd.read_csv("test_data.csv")
target_col = "Bankrupt?"

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

In [48]:
# Iterate through serialized models and calculate all 6 assignment metrics
model_files = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "KNN": "knn.joblib",
    "Naive Bayes": "naive_bayes.joblib",
    "Support Vector Machine": "support_vector_machine.joblib"
}
metrics_list = []

In [49]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)
import joblib

for name, fname in model_files.items():
    model_path = os.path.join("model", fname)
    pipeline = joblib.load(model_path)
    
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    
    metrics_list.append({
        "ML Model Name": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "AUC": round(roc_auc_score(y_test, y_prob), 4),
        "Precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_test, y_pred, zero_division=0), 4),
        "F1": round(f1_score(y_test, y_pred, zero_division=0), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4)
    })

In [50]:
comparison_df = pd.DataFrame(metrics_list)
comparison_df

,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9663,0.8639,0.1806,0.3939,0.2476,0.2515
1,Decision Tree,0.9420,0.6719,0.1008,0.3939,0.1605,0.1776
2,KNN,0.9720,0.7652,0.1845,0.2879,0.2249,0.2168
3,Naive Bayes,0.9516,0.7965,0.1221,0.3939,0.1864,0.2000
4,Support Vector Machine,0.9806,0.8626,0.2642,0.2121,0.2353,0.2270
